# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya - Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset published in Croissant schema format using the `mlcroissant` library. We reference all dataset entities (record sets, fields, columns) by their `@id`.

### Dataset Source
This dataset is published with the Croissant standard and is accessible via schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step will load the Croissant dataset given its URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their constituent fields, referencing everything by `@id`.

We'll inspect record sets, list their corresponding `@id`s, and for each, print its fields (also by `@id`).

In [ ]:
# List available record sets by @id
print('Available record sets:')
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set.id} (name: {record_set.name})")
    record_set_ids.append(record_set.id)

# For each record set, print its fields by @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord set {record_set.id} fields:")
    for field in record_set.fields:
        print(f"  - {field.id} (name: {field.name}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis. Each dataframe is keyed by the record set `@id`. We'll also display the first few rows and columns for one record set as example.

In [ ]:
# Extract data from each record set identified above
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set: {record_set_id}")

# Choose first available record set for continued analysis
if dataframes:
    main_record_set_id = next(iter(dataframes.keys()))
    df = dataframes[main_record_set_id]  # Use first as example
else:
    main_record_set_id = None
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing logic, such as filtering on a numeric field, normalizing, or grouping, referencing all entities by their `@id`.

In [ ]:
# Select a numeric field by its @id for analysis
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Attempt to guess a numeric field by looking at columns and dtypes
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field detected. Please update numeric_field manually if needed.")
    else:
        print(f"Using numeric field (by @id): {numeric_field}")

        # Example threshold for filtering
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Attempt grouping by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (field @id):")
            print(grouped_df.head())
        else:
            print("No suitable field for grouping found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distribution of the selected numeric field, and its relationship with a categorical field (if available), always referencing dataset columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_record_set_id is not None and numeric_field is not None:
    # Histogram of numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if group_field is not None:
        # Boxplot grouped by group_field
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field or main record set for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process a Croissant-structured dataset using `mlcroissant`. All data entities are referenced by their unique `@id`s for transparency and reproducibility. This initial exploration can be extended for advanced statistical analysis or machine learning workflows.

For more information on the dataset and its schema, review the Croissant file at:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)